# Étape 2 : Nettoyage, Standardisation et Clustering non supervisé

Dans cette étape, on prépare les données pour les algorithmes d'apprentissage non supervisé (Clustering : K-Means, DBSCAN), puis on les lance.

**Objectifs :**
1. **Nettoyage** : S'assurer qu'il n'y a plus de valeurs manquantes et isoler les labels (Résultat COVID) pour ne pas tricher lors du clustering.
2. **Gestion de l'Âge** : Éviter que l'âge (seule variable continue) n'écrase les autres variables dans le calcul des distances. On le transforme en 4 variables binaires (One-Hot Encoding).
3. **Standardisation binaire** : Ramener toutes les comorbidités (actuellement codées 1=Oui, 2=Non) à un format binaire classique (1=Oui, 0=Non).
4. **Clustering** : K-Means + DBSCAN sur les profils cliniques, sans jamais regarder le diagnostic COVID.
5. **Interprétation** : Comparer les clusters trouvés aux vrais labels pour voir si l'algo a naturellement séparé les patients COVID+ des COVID-.

À la fin, notre dataset sera composé **uniquement de 0 et de 1**, garantissant un poids parfaitement égal pour chaque caractéristique clinique.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── 1. Chargement des données Silver ──────────────────────────────────────
SILVER_PATH = '../../data/layer_silver_data_transform/covid_cleaned.csv'
df = pd.read_csv(SILVER_PATH)

print(f"Données chargées : {df.shape[0]:,} patients, {df.shape[1]} colonnes.")

### Nettoyage et séparation des labels
Les données provenant de notre couche Silver sont déjà propres (sans `97, 98, 99`). On s'assure de supprimer les éventuelles lignes avec `NaN` par sécurité.
Ensuite, le clustering étant **non supervisé**, nous devons retirer la variable cible (`Resultat_COVID`). Nous la conservons dans une variable à part pour pouvoir analyser nos clusters à l'étape finale.

In [ ]:
# ── Nettoyage et Labels ───────────────────────────────────────────────────
df.dropna(inplace=True)

# Sauvegarde des labels pour l'interprétation finale
labels_covid = df['Resultat_COVID'].copy()

# Suppression de la cible du dataset d'entraînement
df_cluster = df.drop(columns=['Resultat_COVID']).copy()

print("Aperçu des données d'entraînement (sans labels) :")
df_cluster.head(3)

### Gestion de l'âge (Discrétisation et One-Hot Encoding)
Si l'on utilise un `StandardScaler` sur l'âge, on obtiendra des valeurs comme -1.5, 0.2, 2.1, tandis que les comorbidités seront à 0 ou 1. L'âge risquerait encore de dominer légèrement.

La meilleure approche est de **discrétiser** l'âge en tranches (Enfant, Jeune, Adulte, Senior) puis d'appliquer un **One-Hot Encoding**.

In [ ]:
# ── Discrétisation de l'Âge ───────────────────────────────────────────────
bins = [-1, 17, 39, 59, 150]
noms_tranches = ['tranche_age_enfant', 'tranche_age_jeune', 'tranche_age_adulte', 'tranche_age_senior']

df_cluster['Tranche_Age'] = pd.cut(df_cluster['Age'], bins=bins, labels=noms_tranches)
df_cluster = pd.get_dummies(df_cluster, columns=['Tranche_Age'], dtype=int)
df_cluster.drop(columns=['Age'], inplace=True)

print("Colonnes liées à l'âge après encodage :")
print([c for c in df_cluster.columns if 'tranche' in c])

### Standardisation des comorbidités (Ré-encodage 1/2 en 1/0)
Dans le système SISVER mexicain, les réponses binaires sont encodées avec `1 = Oui` et `2 = Non`.
Pour un algorithme mathématique, `0` (Absence) et `1` (Présence) est beaucoup plus sain. On remplace donc tous les `2` par des `0`.

In [ ]:
# ── Standardisation Binaire 0/1 ───────────────────────────────────────────
cols_originales = [c for c in df_cluster.columns if not c.startswith('Tranche_Age_')]
df_cluster[cols_originales] = df_cluster[cols_originales].replace(2, 0)

print("Vérification : min et max de toutes les colonnes (doivent tous être 0 et 1)")
print(df_cluster.describe().loc[['min', 'max']])

### Bilan et Export (Couche Gold)
Notre dataset est maintenant **parfaitement standardisé**. Toutes les variables sont sur une échelle binaire stricte [0, 1].
On exporte ce dataset dans la couche `Gold`.

In [ ]:
# ── Export pour le Clustering (Layer Gold) ────────────────────────────────
import os
gold_dir = '../../data/layer_gold_data_model'
os.makedirs(gold_dir, exist_ok=True)

df_gold = df_cluster.copy()
df_gold['Label_Resultat_COVID'] = labels_covid.values

output_path = f'{gold_dir}/covid_clustering_ready.csv'
df_gold.to_csv(output_path, index=False)

print(f"Dataset Gold exporté : {output_path}")
print(f"Shape : {df_gold.shape[0]:,} patients x {df_gold.shape[1]} colonnes")

---
## Clustering non supervisé — K-Means + DBSCAN

On a maintenant un dataset entièrement binaire (0/1), sans la colonne `Label_Resultat_COVID`.
L'idée c'est de laisser tourner K-Means et DBSCAN sur les **profils cliniques uniquement** et de voir si les groupes qu'ils forment correspondent naturellement aux patients COVID+ et COVID-.

- **Si oui** → les comorbidités seules suffisent à distinguer les deux populations
- **Si non** → ça confirme que le COVID est difficile à prédire par le profil clinique seul (cohérent avec nos modèles supervisés qui plafonnent à F1~0.57)

**Démarche :** On commence par chercher le bon K avec la méthode du coude (estimation visuelle), puis on le confirme précisément avec le Score de Silhouette, avant de lancer K-Means.

In [ ]:
from sklearn.cluster import KMeans, DBSCAN, MiniBatchKMeans
from sklearn.metrics import silhouette_score

# Dataset pour clustering : sans le label
X = df_cluster.copy()

print(f"Dataset clustering : {X.shape[0]:,} patients × {X.shape[1]} features (toutes binaires 0/1)")
print(f"Features : {list(X.columns)}")

### Étape 1 — Méthode du Coude (Elbow Method) : estimation visuelle du bon K

La **méthode du coude** consiste à entraîner K-Means avec K allant de 2 à 10 et tracer l'**inertie** (somme des distances de chaque point à son centroïde). Quand la courbe arrête de descendre fortement, on voit un "coude" — c'est une indication du bon K.

**Attention :** Sur des données binaires (0/1), la courbe est souvent lisse sans coude très marqué. Le coude donne une **fourchette de K candidats**, mais ne suffit pas seul à trancher. C'est pourquoi on utilisera ensuite le Score de Silhouette pour confirmer précisément.

In [ ]:
# ── Méthode du Coude ──────────────────────────────────────────────────────
SAMPLE_ELBOW = 50000
X_sample = X.sample(n=SAMPLE_ELBOW, random_state=42)

inertias = []
K_range = range(2, 11)

print(f"Calcul de l'inertie pour K=2 à 10 (sur {SAMPLE_ELBOW:,} patients)...")
for k in K_range:
    km = MiniBatchKMeans(n_clusters=k, random_state=42, batch_size=5000, n_init=3)
    km.fit(X_sample)
    inertias.append(km.inertia_)
    print(f"  K={k:2d} | Inertie = {km.inertia_:,.0f}")

# Calcul des chutes pour identifier le coude
chutes = [inertias[i-1] - inertias[i] for i in range(1, len(inertias))]
k_coude_approx = list(K_range)[chutes.index(max(chutes)) + 1]
print(f"\nPlus grande chute d'inertie : entre K={k_coude_approx-1} et K={k_coude_approx}")
print(f"-> Le coude se situe approximativement autour de K={k_coude_approx} (à confirmer avec le Silhouette)")

plt.figure(figsize=(10, 5))
plt.plot(list(K_range), inertias, 'bo-', linewidth=2, markersize=8)
plt.axvline(x=k_coude_approx, color='orange', linestyle='--', linewidth=1.5,
            label=f'Coude approximatif (K={k_coude_approx})')
plt.xlabel('Nombre de clusters K', fontsize=12)
plt.ylabel('Inertie (somme des distances au centroïde)', fontsize=12)
plt.title('Méthode du Coude — Estimation visuelle du bon K', fontsize=14, fontweight='bold')
plt.xticks(list(K_range))
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Résultat du coude

La courbe descend progressivement sans coude très marqué — c'est **normal sur des données binaires** : les patients ont des profils assez proches, donc l'inertie ne chute pas brusquement.

Le coude donne une **indication** (autour de K=2 ou 3), mais la courbe reste ambiguë. Pour choisir précisément le bon K, on utilise le **Score de Silhouette** à l'étape suivante, qui est une mesure plus fiable de la qualité réelle des clusters.

### Étape 2 — Score de Silhouette : déterminer précisément K_OPTIMAL

Le **score de Silhouette** mesure à quel point chaque patient est bien assigné à son cluster comparé aux clusters voisins. Il varie de -1 à +1 :
- **> 0.5** → clusters bien séparés
- **0.25 à 0.5** → structure modérée (attendu sur données cliniques)
- **< 0.25** → clusters peu distincts

On teste K=2 à 8 et on choisit le K avec le **meilleur score** — c'est notre `K_OPTIMAL`.
Calcul sur un échantillon de 10 000 patients (O(n²), trop lent sur 260k).

In [ ]:
# ── Score de Silhouette pour K=2 à 8 ─────────────────────────────────────
SAMPLE_SIL = 10000
idx_sil = np.random.RandomState(42).choice(len(X), SAMPLE_SIL, replace=False)
X_sil = X.iloc[idx_sil]

print("Calcul du Score de Silhouette pour K=2 à 8...")
sil_scores = {}
for k in range(2, 9):
    km_tmp = MiniBatchKMeans(n_clusters=k, random_state=42, batch_size=5000, n_init=3)
    labels_tmp = km_tmp.fit_predict(X)
    sil_tmp = silhouette_score(X_sil, labels_tmp[idx_sil])
    sil_scores[k] = sil_tmp
    print(f"  K={k} | Silhouette = {sil_tmp:.4f}")

# K_OPTIMAL = celui avec le meilleur Silhouette
K_OPTIMAL = max(sil_scores, key=sil_scores.get)
print(f"\n→ K_OPTIMAL = {K_OPTIMAL} (Silhouette = {sil_scores[K_OPTIMAL]:.4f}, meilleur score)")

# Graphique
plt.figure(figsize=(9, 4))
plt.bar(list(sil_scores.keys()), list(sil_scores.values()),
        color=['#e74c3c' if k == K_OPTIMAL else '#3498db' for k in sil_scores],
        edgecolor='white')
plt.xlabel('Nombre de clusters K', fontsize=12)
plt.ylabel('Score de Silhouette', fontsize=12)
plt.title(f'Score de Silhouette par K — Optimal : K={K_OPTIMAL}', fontsize=13, fontweight='bold')
plt.xticks(list(sil_scores.keys()))
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### ✅ K_OPTIMAL confirmé par le Silhouette

Le Score de Silhouette donne le meilleur score pour **K_OPTIMAL** (affiché dans la cellule ci-dessus). C'est ce K qui sera utilisé pour K-Means.

**Ce que le score nous dit :** Un score entre 0.25 et 0.5 indique une **structure modérée** — les clusters existent réellement dans les données, mais ils se chevauchent. C'est attendu : les profils cliniques COVID-19 ne sont pas radicalement différents d'un groupe à l'autre.

On utilise **K_OPTIMAL** directement dans la cellule suivante — il n'est pas fixé à la main, il est calculé automatiquement d'après les données.

### Étape 3 — K-Means sur le dataset complet

On entraîne maintenant K-Means avec `K_OPTIMAL` (déterminé par le Silhouette) sur les **260 000 patients complets**.
On utilise `MiniBatchKMeans` — une version optimisée pour les grands datasets qui donne des résultats quasi-identiques à K-Means classique mais beaucoup plus rapidement.

In [ ]:
# ── K-Means sur le dataset complet ────────────────────────────────────────
# K_OPTIMAL est calculé automatiquement par le Silhouette (cellule précédente)
print(f"Entraînement K-Means (K={K_OPTIMAL}) sur {len(X):,} patients...")
kmeans = MiniBatchKMeans(n_clusters=K_OPTIMAL, random_state=42, batch_size=10000, n_init=5)
labels_kmeans = kmeans.fit_predict(X)

print(f"\nDistribution des clusters K-Means :")
for k in range(K_OPTIMAL):
    n = (labels_kmeans == k).sum()
    print(f"  Cluster {k} : {n:,} patients ({n/len(X)*100:.1f}%)")

### Étape 4 — Interprétation des clusters K-Means

On calcule la proportion de `1` pour chaque feature par cluster : c'est le **profil clinique moyen** de chaque groupe.
Ensuite on compare avec les vrais labels COVID pour voir si l'algorithme a naturellement séparé COVID+ et COVID-.

In [ ]:
# ── Profil moyen par cluster (heatmap) ────────────────────────────────────
X_interpret = X.copy()
X_interpret['Cluster_KMeans'] = labels_kmeans
X_interpret['COVID'] = labels_covid.values  # 1=COVID+, 2=COVID-

feature_cols = list(X.columns)
profil = X_interpret.groupby('Cluster_KMeans')[feature_cols].mean()

plt.figure(figsize=(15, max(5, K_OPTIMAL * 1.5)))
sns.heatmap(
    profil.T,
    annot=True, fmt='.2f', cmap='YlOrRd',
    linewidths=0.5, cbar_kws={'label': 'Proportion de patients avec feature=1'}
)
plt.title(f'Profil clinique moyen par cluster K-Means (K={K_OPTIMAL})', fontsize=14, fontweight='bold')
plt.xlabel('Cluster')
plt.ylabel('Feature clinique')
plt.tight_layout()
plt.show()

In [ ]:
# ── Clusters vs vrais labels COVID ────────────────────────────────────────
print("=" * 55)
print("RÉSULTAT CLEF : Clusters K-Means vs Vrais Labels COVID")
print("=" * 55)
print()

global_pct = (labels_covid == 1).mean() * 100
print(f"Référence globale : {global_pct:.1f}% COVID+ dans le dataset")
print()

pct_covid_par_cluster = []
for k in range(K_OPTIMAL):
    subset = X_interpret[X_interpret['Cluster_KMeans'] == k]
    n_total = len(subset)
    pct_pos = (subset['COVID'] == 1).mean() * 100
    pct_covid_par_cluster.append(pct_pos)
    ecart = pct_pos - global_pct
    sens = f"+{ecart:.1f}pp (plus de COVID+)" if ecart > 0 else f"{ecart:.1f}pp (moins de COVID+)"
    print(f"  Cluster {k} ({n_total:,} patients) : {pct_pos:.1f}% COVID+ | {sens}")

# Tableau croisé
print()
crosstab = pd.crosstab(
    X_interpret['Cluster_KMeans'],
    X_interpret['COVID'].map({1: 'COVID+', 2: 'COVID-'}),
    normalize='index'
).round(3) * 100
print("Tableau croisé (% par cluster) :")
print(crosstab.to_string())

In [ ]:
# ── Visualisation ─────────────────────────────────────────────────────────
cluster_sizes = [(labels_kmeans == k).sum() for k in range(K_OPTIMAL)]
colors = plt.cm.tab10.colors[:K_OPTIMAL]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# % COVID+ par cluster
bars = axes[0].bar(range(K_OPTIMAL), pct_covid_par_cluster, color=colors, edgecolor='white', linewidth=1.5)
axes[0].axhline(y=global_pct, color='black', linestyle='--', linewidth=1.5,
                label=f'Moyenne globale ({global_pct:.1f}%)')
axes[0].set_xlabel('Cluster K-Means', fontsize=12)
axes[0].set_ylabel('% COVID+', fontsize=12)
axes[0].set_title(f'Proportion COVID+ par cluster (K={K_OPTIMAL})', fontsize=13, fontweight='bold')
axes[0].set_xticks(range(K_OPTIMAL))
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)
for i, v in enumerate(pct_covid_par_cluster):
    axes[0].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontweight='bold')

# Taille des clusters
axes[1].pie(
    cluster_sizes,
    labels=[f'Cluster {k}\n({cluster_sizes[k]:,})' for k in range(K_OPTIMAL)],
    colors=colors, autopct='%1.1f%%', startangle=90
)
axes[1].set_title('Taille des clusters K-Means', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

### ✅ Ce que les clusters représentent

Chaque cluster regroupe des patients aux **profils cliniques similaires**. En regardant le graphique ci-dessus, on peut observer :
- Certains clusters ont un taux de COVID+ **nettement au-dessus** de la moyenne globale (~39%) → profils à risque élevé
- D'autres sont **en-dessous** → profils moins touchés
- Mais aucun cluster ne concentre que des COVID+ ou que des COVID-

**Conclusion partielle :** Le profil clinique crée des groupes avec des **niveaux de risque différents**, mais ne sépare pas proprement les deux populations. C'est cohérent avec la difficulté du problème de classification (F1~0.57 à l'Étape 3).

### Étape 5 — DBSCAN : clustering par densité

Contrairement à K-Means, **DBSCAN** n'a pas besoin qu'on fixe K à l'avance. Il trouve lui-même le nombre de clusters en regroupant les points **denses** (proches les uns des autres). Les points isolés sont classés comme **bruit** (label -1).

On utilise la distance de **Hamming** (proportion de features différentes entre deux patients) — adaptée aux données binaires.
On travaille sur 30 000 patients car DBSCAN est lent en O(n²).

In [ ]:
# ── DBSCAN sur sous-échantillon ───────────────────────────────────────────
SAMPLE_DBSCAN = 30000
X_dbscan = X.sample(n=SAMPLE_DBSCAN, random_state=42)
labels_covid_dbscan = labels_covid.loc[X_dbscan.index]

print(f"DBSCAN sur {SAMPLE_DBSCAN:,} patients (metric=hamming, eps=0.35, min_samples=15)...")
db = DBSCAN(eps=0.35, min_samples=15, metric='hamming', n_jobs=-1)
labels_dbscan = db.fit_predict(X_dbscan)

n_clusters_db = len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
n_noise = (labels_dbscan == -1).sum()

print(f"\nRésultats DBSCAN :")
print(f"  Clusters trouvés : {n_clusters_db}")
print(f"  Points aberrants (bruit) : {n_noise:,} ({n_noise/SAMPLE_DBSCAN*100:.1f}%)")
print()

X_dbscan_copy = X_dbscan.copy()
X_dbscan_copy['Cluster_DBSCAN'] = labels_dbscan
X_dbscan_copy['COVID'] = labels_covid_dbscan.values

ref_db = (labels_covid_dbscan == 1).mean() * 100
for label in sorted(set(labels_dbscan)):
    subset = X_dbscan_copy[X_dbscan_copy['Cluster_DBSCAN'] == label]
    pct_pos = (subset['COVID'] == 1).mean() * 100
    nom = f"Cluster {label}" if label != -1 else "Bruit (hors cluster)"
    print(f"  {nom} ({len(subset):,} patients) : {pct_pos:.1f}% COVID+")

print(f"\n  Référence globale (échantillon) : {ref_db:.1f}% COVID+")

### ✅ Résultat DBSCAN : nuage dense homogène

DBSCAN a regroupé l'ensemble des patients dans **un seul cluster dense**, sans aucun point aberrant (bruit = 0%).

**Ce que ça signifie :** Avec une tolérance de 35% de features différentes (distance Hamming = 0.35), tous les patients sont considérés comme "proches" — il n'y a pas de petits groupes isolés aux profils vraiment atypiques. Les profils cliniques COVID-19 forment un **continuum dense**, pas des îlots séparés.

Ce résultat est cohérent avec K-Means : il n'y a pas de séparation naturelle et nette entre COVID+ et COVID-.

---
## Conclusion et Décision — Que fait-on de ce clustering ?

### Ce qu'on a appris

| Algo | Résultat | Interprétation |
|------|----------|----------------|
| **Méthode du coude** | Coude approximatif, courbe lisse | Normal sur données binaires — on s'appuie sur le Silhouette |
| **Silhouette** | K_OPTIMAL = meilleur score entre K=2 et 8 | Clusters réels mais qui se chevauchent (score modéré 0.25–0.50) |
| **K-Means (K=K_OPTIMAL)** | Clusters avec des taux COVID+ différents | Signal faible mais réel : comorbidités ↔ risque COVID |
| **DBSCAN** | 1 seul cluster, 0 bruit | Tous les profils forment un nuage continu et dense |

### Décision : que fait-on de ces clusters ?

**On garde les étiquettes originales COVID+/COVID- pour la suite.**

Le clustering ici joue un rôle **exploratoire** — il nous aide à comprendre la structure interne des données. Ce n'est pas un outil de prédiction, et il ne remplace pas les étiquettes médicales réelles.

Concrètement :
- **On n'utilise PAS les numéros de cluster** pour prédire le COVID à l'Étape 3 — la variable cible reste `Label_Resultat_COVID` (1=COVID+, 0=COVID-).
- **Les étiquettes originales sont les seules fiables** : elles viennent de vrais tests médicaux. Un numéro de cluster, lui, est construit par l'algorithme sans jamais avoir vu le diagnostic.
- **Le clustering confirme** que le COVID est difficile à prédire par le profil clinique seul, ce qui justifie nos choix méthodologiques à l'Étape 3 (maximiser le Recall, utiliser des modèles complexes comme XGBoost).

### Ce qu'on aurait pu faire de plus (piste non retenue)

On aurait pu ajouter le **numéro de cluster comme feature supplémentaire** dans le modèle supervisé (feature engineering). L'idée : donner au modèle de l'Étape 3 une information sur le "groupe de profil" du patient. En pratique, les clusters se chevauchent trop pour que cela apporte un gain significatif — on a choisi de ne pas le faire pour garder le modèle simple et interprétable.